# SentenceDebias on mBERT (English direction vs Urdu direction)


In [1]:
!pip -q install transformers datasets scikit-learn scipy tqdm


In [2]:
import sys, glob, os
DATA = "/kaggle/input/datasets/shanzaysaleem/nlp-project"
OUT = "m3_results"

roots = [DATA, ".", "/kaggle/input"]
def locate(name):
    for root in roots:
        matches = glob.glob(os.path.join(root, "**", name), recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"{name} not found under {roots}")

sys.path.insert(0, os.path.dirname(locate("sentence_debias.py")))
import torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM
import sentence_debias as sd

N_SENT = 10_000
K = 1                    # PCA components (try 2 as a sensitivity check)
POOL = "mean"            # "mean" or "cls" (sensitivity check)
SEED = 0                 # Wikipedia sampling seed (run 0,1,2 and compare)
AGREEMENT_FILTER = False # True = drop Urdu sentences with agreement-bearing tokens
                         # (avoids ungrammatical noun-only counterfactuals)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
tokenizer = AutoTokenizer.from_pretrained(sd.MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(sd.MODEL_NAME).to(device).eval()


device: cuda


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  672MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Evaluation data (same sets as Baseline)

In [3]:
ur = pd.read_csv(locate("crows_pairs_gender_ur_eval.csv"), encoding="utf-8-sig")
en = pd.read_csv(locate("english_gender_all_results.csv"))
cols = ["pair_id", "sent_more", "sent_less", "stereo_antistereo"]
eval_sets = {
    "English - 262": en[cols],
    "English - 242 aligned": en[en.pair_id.isin(ur["id"])][cols],
    "Urdu - 242": ur.rename(columns={"id": "pair_id", "sent_more_ur": "sent_more",
                                        "sent_less_ur": "sent_less"})[cols],
}
print({k: len(v) for k, v in eval_sets.items()})


{'English - 262': 262, 'English - 242 aligned': 242, 'Urdu - 242': 242}


## 2. Gender words


In [4]:
EN_NO_PRONOUNS = False
en_pairs = [p for p in sd.EN_PAIRS if not (EN_NO_PRONOUNS and (p[0] in sd.EN_PRONOUN_WORDS))]
sw_en = sd.GenderSwapper(en_pairs)
sw_ur = sd.GenderSwapper(sd.load_urdu_pairs(locate("gender_word_pairs_ur.csv")))
print(len(en_pairs), "English pairs;", len(sw_ur.map)//2, "Urdu pairs")
print(sw_ur.swap("وہ مرد گیا اور طالب علم آیا"))

print("excluded name/honorific terms:", sorted(sd.EXCLUDE_UR))


39 English pairs; 34 Urdu pairs
وہ عورت گیا اور طالبہ آیا
excluded name/honorific terms: ['ابو', 'جناب', 'حاجی', 'دوست', 'سہیلی', 'صاحب', 'مالک']


## 3. Wikipedia sentences containing a gender word (streamed — no full download)

In [5]:
import re
ur_reject = sd.UR_AGREEMENT_RX if AGREEMENT_FILTER else None
en_sents, en_counts = sd.collect_sentences(sd.iter_wikipedia("en", 300_000, seed=SEED), "en", sw_en, n_target=N_SENT, seed=SEED)
ur_sents, ur_counts = sd.collect_sentences(sd.iter_wikipedia("ur", 300_000, seed=SEED), "ur", sw_ur, n_target=N_SENT, seed=SEED, reject_regex=ur_reject)
print(len(en_sents), "English /", len(ur_sents), "Urdu sentences  (seed", SEED, ", agreement filter", AGREEMENT_FILTER, ")")
print("Urdu per-term counts:", dict(sorted(ur_counts.items(), key=lambda x: -x[1])))
print("EXAMPLE:", ur_sents[0], "\n     ->", sw_ur.swap(ur_sents[0]))

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

10000 English / 10000 Urdu sentences  (seed 0 , agreement filter False )
Urdu per-term counts: {'شاعر': 400, 'بادشاہ': 400, 'مصنف': 400, 'اداکارہ': 400, 'والد': 400, 'بیٹا': 400, 'بھائی': 400, 'ماں': 400, 'عورت': 400, 'اداکار': 400, 'ملکہ': 400, 'باپ': 400, 'آدمی': 400, 'بیوی': 400, 'مرد': 400, 'والدہ': 400, 'خاتون': 366, 'بیٹی': 361, 'استاد': 328, 'شہزادہ': 225, 'بہن': 223, 'چچا': 212, 'شوہر': 191, 'لڑکی': 184, 'گلوکار': 181, 'ملازم': 139, 'دادا': 131, 'طالب علم': 114, 'گلوکارہ': 103, 'لڑکا': 80, 'پڑوسی': 76, 'شہزادی': 72, 'مصنفہ': 53, 'نانا': 52, 'ماموں': 48, 'داماد': 44, 'شاعرہ': 38, 'بوڑھا': 34, 'پوتا': 30, 'بہو': 30, 'دلہن': 24, 'نوکر': 23, 'خالہ': 21, 'نند': 19, 'بوڑھی': 19, 'بھتیجا': 19, 'سسر': 16, 'پوتی': 16, 'ساس': 15, 'پھوپھی': 15, 'نانی': 14, 'دادی': 14, 'نواسی': 10, 'طالبہ': 9, 'چچی': 8, 'دیور': 7, 'نوکرانی': 6, 'ملازمہ': 4, 'بھتیجی': 4, 'بھانجی': 4, 'دولہا': 4, 'پڑوسن': 4, 'خالو': 3, 'پھوپھا': 2, 'استانی': 2, 'ممانی': 1, 'بھانجا': 1, 'نواسا': 1}
EXAMPLE: اس قلیل آمدنی سے ا

## 4. Directions + before/after scoring


In [6]:
import importlib
importlib.reload(sd)

summary, dirs, pair_level = sd.run_experiment(
    model, tokenizer, device, en_sents, ur_sents, eval_sets, OUT,
    en_swapper=sw_en, ur_swapper=sw_ur, k=K, pool=POOL,
    baseline_summary_csv=locate("baseline_summary.csv"),
)
display(summary)
paired = pd.read_csv(f"{OUT}/paired_comparisons.csv")
display(paired)

en direction: 10000 sentence pairs, explained variance [0.393]
ur direction: 10000 sentence pairs, explained variance [0.139]
{
  "cosine_en_vs_ur_top_component": -0.6413323283195496,
  "abs_cosine_en_vs_ur": 0.6413323283195496,
  "explained_en": [
    0.3934410810470581
  ],
  "explained_ur": [
    0.13929376006126404
  ],
  "n_pairs_en": 10000,
  "n_pairs_ur": 10000,
  "pool": "mean",
  "k": 1,
  "swap_diff_along_en_dir__on_en_sentences": 0.5765476822853088,
  "swap_diff_along_en_dir__on_ur_sentences": 0.1805189996957779,
  "swap_diff_along_ur_dir__on_en_sentences": 0.370206356048584,
  "swap_diff_along_ur_dir__on_ur_sentences": 0.27328670024871826
}


100%|██████████| 242/242 [00:38<00:00,  6.34it/s]

baseline check  English - 262            mine= 50.38  member2= 50.38  OK
baseline check  English - 242 aligned    mine= 50.00  member2= 50.00  OK
baseline check  Urdu - 242               mine= 46.69  member2= 46.69  OK


,dataset,condition,evaluated_pairs,stereotype_preference_pct,ci95_lo,ci95_hi,anti_stereotype_preference_pct,ties_pct,deviation_from_50
0,English - 262,before (original mBERT),262,50.38,44.6,56.5,49.24,0.38,0.38
1,English - 242 aligned,before (original mBERT),242,50.00,43.8,56.2,49.59,0.41,0.00
2,Urdu - 242,before (original mBERT),242,46.69,40.9,53.3,53.31,0.00,3.31
3,English - 262,after: English direction,262,53.44,47.7,59.5,46.56,0.00,3.44
4,English - 242 aligned,after: English direction,242,53.31,47.1,59.5,46.69,0.00,3.31
5,Urdu - 242,after: English direction,242,48.35,42.6,55.0,51.65,0.00,1.65
6,English - 262,after: Urdu direction,262,51.91,46.2,58.0,48.09,0.00,1.91
7,English - 242 aligned,after: Urdu direction,242,51.65,45.5,57.9,48.35,0.00,1.65
8,Urdu - 242,after: Urdu direction,242,48.76,43.0,55.4,51.24,0.00,1.24


,dataset,condition,n_paired,delta_stereotype_pp,delta_ci95_lo,delta_ci95_hi,changed_toward_stereotype,changed_away_from_stereotype,paired_p_value
0,English - 262,after: English direction,262,3.05,-0.76,6.49,16,8,0.151590
1,English - 262,after: Urdu direction,262,1.53,-0.76,3.82,7,3,0.343750
2,English - 242 aligned,after: English direction,242,3.31,-0.41,7.02,15,7,0.133801
3,English - 242 aligned,after: Urdu direction,242,1.65,-0.83,4.14,7,3,0.343750
4,Urdu - 242,after: English direction,242,1.65,-1.25,4.96,10,6,0.454498
5,Urdu - 242,after: Urdu direction,242,2.07,-1.24,5.79,12,7,0.359283


In [7]:
import shutil; shutil.make_archive("m3_results", "zip", OUT)
print("saved m3_results.zip in", os.getcwd(), "-")

saved m3_results.zip in /kaggle/working -


In [8]:
# Interpretation generated from the results (so it cannot go stale after a rerun)
before = summary[summary.condition.str.startswith("before")].set_index("dataset")
for _, r in paired.iterrows():
    b = before.loc[r.dataset]
    a = summary[(summary.dataset == r.dataset) & (summary.condition == r.condition)].iloc[0]
    sig = "significant" if r.paired_p_value < 0.05 else "NOT significant"
    base_ci_has_50 = b.ci95_lo <= 50 <= b.ci95_hi
    print(f"{r.dataset:22s} | {r.condition:26s} | {b.stereotype_preference_pct:5.2f}% -> "
          f"{a.stereotype_preference_pct:5.2f}%  (deviation from 50: {b.deviation_from_50:.2f} -> {a.deviation_from_50:.2f}) "
          f"| paired p={r.paired_p_value:.3f} ({sig}); baseline CI contains 50: {base_ci_has_50}")
print("\n|cos(English dir, Urdu dir)| =", round(abs(dirs['en']['direction'][0] @ dirs['ur']['direction'][0]), 3))

English - 262          | after: English direction   | 50.38% -> 53.44%  (deviation from 50: 0.38 -> 3.44) | paired p=0.152 (NOT significant); baseline CI contains 50: True
English - 262          | after: Urdu direction      | 50.38% -> 51.91%  (deviation from 50: 0.38 -> 1.91) | paired p=0.344 (NOT significant); baseline CI contains 50: True
English - 242 aligned  | after: English direction   | 50.00% -> 53.31%  (deviation from 50: 0.00 -> 3.31) | paired p=0.134 (NOT significant); baseline CI contains 50: True
English - 242 aligned  | after: Urdu direction      | 50.00% -> 51.65%  (deviation from 50: 0.00 -> 1.65) | paired p=0.344 (NOT significant); baseline CI contains 50: True
Urdu - 242             | after: English direction   | 46.69% -> 48.35%  (deviation from 50: 3.31 -> 1.65) | paired p=0.454 (NOT significant); baseline CI contains 50: True
Urdu - 242             | after: Urdu direction      | 46.69% -> 48.76%  (deviation from 50: 3.31 -> 1.24) | paired p=0.359 (NOT significant)

## Results

We applied SentenceDebias to mBERT with a gender direction computed from English Wikipedia and from Urdu Wikipedia, and evaluated on CrowS-Pairs (gender). A score of 50% means no preference for the stereotypical sentence.

| Dataset | Original | English direction | Urdu direction |
|---|---:|---:|---:|
| Urdu (242 pairs) | 46.7% | 48.4% | 48.8% |
| English (262 pairs) | 50.4% | 53.4% | 51.9% |


- Urdu moved closer to 50% with both directions, and the Urdu-derived direction was slightly closer (48.8% vs 48.4%).
- None of the changes was statistically significant (paired p ≥ 0.13). The original Urdu model was also not significantly different from 50% (95% CI 40.9–53.3).
- English moved slightly away from 50%, which is the opposite of the intended effect.

